In [1]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

name, memory.total [MiB], memory.used [MiB]
Tesla T4, 15360 MiB, 0 MiB


In [2]:
!pip install -q vllm
!pip uninstall -y -q torchaudio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.0/316.0 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 76.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 52.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.7/322.7 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 72.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.6/782.6 kB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 84.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/43.2 MB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 728.5

In [3]:
import ctypes, glob, os, time
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"
os.environ["VLLM_LOGGING_LEVEL"] = "DEBUG"
_cudart_paths = glob.glob("/usr/**/libcudart.so*", recursive=True)
_loaded = [ctypes.CDLL(p, mode=ctypes.RTLD_GLOBAL) for p in _cudart_paths]
print("preloaded:", _cudart_paths)

preloaded: ['/usr/local/lib/python3.13/dist-packages/nvidia/cuda_runtime/lib/libcudart.so.12', '/usr/local/lib/python3.13/dist-packages/nvidia/cu13/lib/libcudart.so.13', '/usr/local/cuda-12.8/lib64/libcudart.so.12', '/usr/local/cuda-12.8/lib64/libcudart.so.12.8.90', '/usr/local/cuda-12.8/lib64/libcudart.so', '/usr/local/cuda-12.8/targets/x86_64-linux/lib/libcudart.so.12', '/usr/local/cuda-12.8/targets/x86_64-linux/lib/libcudart.so.12.8.90', '/usr/local/cuda-12.8/targets/x86_64-linux/lib/libcudart.so', '/usr/local/cuda/lib64/libcudart.so.12', '/usr/local/cuda/lib64/libcudart.so.12.8.90', '/usr/local/cuda/lib64/libcudart.so', '/usr/local/cuda/targets/x86_64-linux/lib/libcudart.so.12', '/usr/local/cuda/targets/x86_64-linux/lib/libcudart.so.12.8.90', '/usr/local/cuda/targets/x86_64-linux/lib/libcudart.so', '/usr/local/cuda-12/lib64/libcudart.so.12', '/usr/local/cuda-12/lib64/libcudart.so.12.8.90', '/usr/local/cuda-12/lib64/libcudart.so', '/usr/local/cuda-12/targets/x86_64-linux/lib/libcuda

In [6]:
from vllm import LLM, SamplingParams
llm = LLM(model="TinyLlama/TinyLlama-1.1B-Chat-v1.0", gpu_memory_utilization=0.85, max_model_len=2048, enable_prefix_caching=True)
print("model loaded")

INFO 09-12 16:31:38 [entrypoints/.../utils/api_utils.py:286] non-default args: {'max_model_len': 2048, 'enable_prefix_caching': True, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'model': 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'}
WARNING 09-12 16:31:38 [engine/arg_utils.py:1801] The global random seed is set to 0. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.
INFO 09-12 16:31:38 [config/model.py:684] Resolved architecture: LlamaForCausalLM
WARNING 09-12 16:31:38 [config/model.py:2302] Your device 'Tesla T4' (with compute capability 7.5) doesn't support torch.bfloat16. Falling back to torch.float16 for compatibility.
WARNING 09-12 16:31:38 [config/model.py:2355] Casting torch.bfloat16 to torch.float16.
INFO 09-12 16:31:38 [config/model.py:2021] Using max model len 2048
DEBUG 09-12 16:31:38 [config/model.py:2086] Generative models support chunked prefill.
DEBUG 09-12 16:31:38 [config/model.py:214

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

... (141 outputs left out)

DEBUG 09-12 16:33:01 [v1/worker/gpu/cudagraph_utils.py:358] CG Capture: mode=FULL, batch_desc=BatchExecutionDescriptor(cg_mode=<CUDAGraphMode.FULL: 2>, num_tokens=2, num_reqs=2, uniform_token_count=1, max_query_len=None, num_active_loras=0)
DEBUG 09-12 16:33:02 [v1/worker/gpu/cudagraph_utils.py:358] CG Capture: mode=FULL, batch_desc=BatchExecutionDescriptor(cg_mode=<CUDAGraphMode.FULL: 2>, num_tokens=1, num_reqs=1, uniform_token_count=1, max_query_len=None, num_active_loras=0)


Capturing CUDA graphs (FULL): 100%|██████████| 35/35 [00:01<00:00, 23.48it/s]

INFO 09-12 16:33:02 [v1/worker/gpu/model_runner.py:960] Graph capturing finished in 6 secs, took 0.14 GiB
INFO 09-12 16:33:02 [v1/worker/gpu_worker.py:797] CUDA graph pool memory: 0.14 GiB (actual), 0.3 GiB (estimated), difference: 0.16 GiB (120.0%).
INFO 09-12 16:33:02 [v1/worker/gpu_worker.py:860] Free memory on device (14.46/14.56 GiB) on startup. Desired GPU memory utilization is (0.85, 12.38 GiB). Actual usage is 2.19 GiB for consumed memory (weights + non-torch), 0.65 GiB for peak activation, and 0.14 GiB for CUDAGraph memory. Replace gpu_memory_utilization config with `--kv-cache-memory=9936074445` (9.25 GiB) to fit into requested memory, or `--kv-cache-memory=12171667968` (11.34 GiB) to fully utilize gpu memory. Current kv cache memory in use is 9.54 GiB.


DEBUG 09-12 16:33:10 [utils/jit_monitor.py:321] CuTeDSL is not available; skipping CuTeDSL JIT monitor.
INFO 09-12 16:33:11 [utils/jit_monitor.py:85] Kernel JIT monitor activated; monitored JIT compilations during inference will use mode=warn.
DEBUG 09-12 16:33:12 [utils/gc_utils.py:40] GC Debug Config. enabled:False,top_objects:-1
INFO 09-12 16:33:12 [v1/engine/core.py:361] init engine (profile, create kv cache, warmup model) took 50.45 s (compilation: 18.95 s)
DEBUG 09-12 16:33:12 [v1/engine/core.py:215] Batch queue is enabled with size 2
DEBUG 09-12 16:33:12 [utils/gc_utils.py:40] GC Debug Config. enabled:False,top_objects:-1
DEBUG 09-12 16:33:12 [renderers/base.py:258] Warming up chat template processing...
DEBUG 09-12 16:33:12 [transformers_utils/repo_utils.py:326] File or repository not found in hf_hub_download: 404 Client Error. (Request ID: Root=1-6aa57ec8-5f2d882024fb84787606cbaa;a28aad79-20e8-4c54-be77-90edd968526e)
DEBUG 09-12 16:33:12 [transformers_utils/repo_utils.py:326] 

In [9]:
import time
sp = SamplingParams(temperature=0.0, max_tokens=128, min_tokens=128)
animals = ["fox","rabbit","owl","bear","deer","wolf","otter","hawk","lynx","moose","raven","hare","badger","heron","elk","mink","crow","seal","lark","stoat","vole","toad","newt","gull","dove","finch","wren","swan","stork","crane"]
prompts_pool = [f"Write a short story about a {a} exploring a forest." for a in animals]
print(len(prompts_pool), "prompts ready")

30 prompts ready


In [10]:
def run_batch(n): p = prompts_pool[:n]; t0 = time.time(); outs = llm.generate(p, sp); dt = time.time() - t0; toks = sum(len(o.outputs[0].token_ids) for o in outs); return dt, toks
results = []
batch_sizes = [1, 2, 4, 8, 16, 30]
for n in batch_sizes: dt, toks = run_batch(n); thpt = toks/dt; lat = dt/n; results.append((n, dt, toks, thpt, lat)); print(f"n={n:2d}  time={dt:6.2f}s  out_tokens={toks:5d}  throughput={thpt:6.1f} tok/s  avg_latency/req={lat:5.3f}s")

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 1/1 [00:02<00:00,  2.01s/it, est. speed input: 6.99 toks/s, output: 63.94 toks/s]

n= 1  time=  2.03s  out_tokens=  128  throughput=  63.2 tok/s  avg_latency/req=2.027s


Rendering prompts:   0%|          | 0/2 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 2/2 [00:01<00:00,  1.36it/s, est. speed input: 19.04 toks/s, output: 174.05 toks/s]

n= 2  time=  1.54s  out_tokens=  256  throughput= 166.8 tok/s  avg_latency/req=0.768s


... (8 outputs left out)

n=16  time=  1.70s  out_tokens= 2048  throughput=1206.2 tok/s  avg_latency/req=0.106s


Rendering prompts:   0%|          | 0/30 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 30/30 [00:02<00:00, 13.65it/s, est. speed input: 190.77 toks/s, output: 1748.33 toks/s]

n=30  time=  2.26s  out_tokens= 3840  throughput=1696.6 tok/s  avg_latency/req=0.075s


In [11]:
shared_prefix = ("You are a knowledgeable and helpful research assistant with deep expertise in computer science, distributed systems, and machine learning infrastructure. " * 20)
sp_ttft = SamplingParams(temperature=0.0, max_tokens=1, min_tokens=1)
t0 = time.time(); _ = llm.generate([shared_prefix + " Question: What is 2+2?"], sp_ttft); cold = time.time() - t0
t0 = time.time(); _ = llm.generate([shared_prefix + " Question: What is the capital of France?"], sp_ttft); warm1 = time.time() - t0
t0 = time.time(); _ = llm.generate([shared_prefix + " Question: Name a primary color."], sp_ttft); warm2 = time.time() - t0
print(f"prefix length (chars): {len(shared_prefix)}")
print(f"cold (no cache hit)      : {cold:.3f}s")
print(f"warm (prefix cache hit) 1: {warm1:.3f}s  speedup={cold/warm1:.2f}x")
print(f"warm (prefix cache hit) 2: {warm2:.3f}s  speedup={cold/warm2:.2f}x")

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  5.61it/s, est. speed input: 3024.50 toks/s, output: 5.69 toks/s]


Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 1/1 [00:00<00:00, 16.80it/s, est. speed input: 9041.65 toks/s, output: 17.02 toks/s]


Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 1/1 [00:00<00:00, 10.16it/s, est. speed input: 5505.03 toks/s, output: 10.40 toks/s]

prefix length (chars): 3080
cold (no cache hit)      : 0.206s
warm (prefix cache hit) 1: 0.125s  speedup=1.65x
warm (prefix cache hit) 2: 0.135s  speedup=1.53x
